In [32]:
from hana_ml import dataframe
cc = dataframe.ConnectionContext(userkey='MyDBKey')

In [33]:
import numpy as np
import pandas as pd
np.random.seed(2023)
seq_len = 16
data = pd.concat((pd.DataFrame(dict(DATE=pd.date_range('2025-01-01', periods=seq_len),
                                    ID=range(seq_len))),
                  pd.DataFrame(10 * np.random.normal(size=(seq_len, 2)), columns=['X1', 'X2'])),
                  axis=1)

In [34]:
from hana_ml.dataframe import create_dataframe_from_pandas
sim_df = create_dataframe_from_pandas(cc, data,
                                      'SIM_CORR_DATA_TBL',
                                      force=True)

100%|██████████| 1/1 [00:00<00:00,  2.73it/s]


In [35]:
from hana_ai.tools.hana_ml_tools.correlation_tools import Correlation
cf_tool = Correlation(cc)

In [36]:
from langchain.agents import initialize_agent, AgentType
from gen_ai_hub.proxy.langchain import init_llm
llm = init_llm('gpt-4o', temperature=0.0, max_tokens=2000) # used to do logical reasoning
tools = [cf_tool] # Add any tools here
agent_chain = initialize_agent(tools, llm, agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION)

### Test 1 : Timestamp Index

In [8]:
tool_input = dict(table_name='SIM_CORR_DATA_TBL',
                  key='DATE',
                  x='X1',
                  calculate_pacf=True)
                  #y='X2',
                  #max_lag=10)
cf_tool.run(tool_input=tool_input)

'{"correlation_result_table": "SIM_CORR_DATA_TBL_CORRELATION_RESULT"}'

In [10]:
cc.table("SIM_CORR_DATA_TBL_CORRELATION_RESULT").head(3).collect()

,LAG,CV,CF,PACF
0,0,121.881690,1.000000,1.000000
1,1,29.159637,0.239245,0.239245
2,2,-14.649828,-0.120197,-0.188208


In [11]:
instruction = "Please compute the autocorrelation of the time-series data in column X3 of table " +\
"SIM_CORR_DATA_TBL along with its partial auto-correlation coefficients, where key is DATE"
agent_chain.invoke(instruction)

{'input': 'Please compute the autocorrelation of the time-series data in column X3 of table SIM_CORR_DATA_TBL along with its partial auto-correlation coefficients, where key is DATE',
 'output': "It seems there was an error because the column 'X3' does not exist in the table 'SIM_CORR_DATA_TBL'. Please verify the column name and try again. If you have the correct column name, I can proceed with the computation."}

In [13]:
cc.table("SIM_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF
0,0,121.881690,1.000000,1.000000
1,1,29.159637,0.239245,0.239245
2,2,-14.649828,-0.120197,-0.188208
3,3,-12.993741,-0.106609,-0.030145
4,4,-5.505351,-0.045170,-0.032853


### Test 2 : ACFs with Confidence Interval

In [13]:
tool_input = dict(table_name='SIM_CORR_DATA_TBL',
                  key='ID',
                  x='X1',
                  calculate_confint=True)
                  #max_lag=10)
cf_tool.run(tool_input=tool_input)

'{"correlation_result_table": "SIM_CORR_DATA_TBL_CORRELATION_RESULT"}'

In [14]:
cc.table("SIM_CORR_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF,ACF_CONFIDENCE_BOUND,PACF_CONFIDENCE_BOUND
0,0,121.881690,1.000000,1.000000,NaN,NaN
1,1,29.159637,0.239245,0.239245,0.489991,0.489991
2,2,-14.649828,-0.120197,-0.188208,0.517278,0.489991
3,3,-12.993741,-0.106609,-0.030145,0.523940,0.489991
4,4,-5.505351,-0.045170,-0.032853,0.529123,0.489991


In [15]:
instruction = "Please compute the autocorrelation of the time-series data in column X1 of table " +\
"SIM_CORR_DATA_TBL along with its partial auto-correlation coefficients as well as confidence intervals, " +\
"where key is DATE"
agent_chain.invoke(instruction)

{'input': 'Please compute the autocorrelation of the time-series data in column X1 of table SIM_CORR_DATA_TBL along with its partial auto-correlation coefficients as well as confidence intervals, where key is DATE',
 'output': 'The autocorrelation, partial autocorrelation coefficients, and confidence intervals for the time-series data in column X1 of table SIM_CORR_DATA_TBL have been computed and stored in the table SIM_CORR_DATA_TBL_CORRELATION_RESULT.'}

In [16]:
cc.table("SIM_CORR_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF,ACF_CONFIDENCE_BOUND,PACF_CONFIDENCE_BOUND
0,0,121.881690,1.000000,1.000000,NaN,NaN
1,1,29.159637,0.239245,0.239245,0.489991,0.489991
2,2,-14.649828,-0.120197,-0.188208,0.517278,0.489991
3,3,-12.993741,-0.106609,-0.030145,0.523940,0.489991
4,4,-5.505351,-0.045170,-0.032853,0.529123,0.489991


### Case 3 : Correlation between Two Columns

In [18]:
instruction = "Please compute the correlation between column X1 and column X2 in table SIM_CORR_DATA_TBL, "+\
", where key is ID and maximum lag is 3."
agent_chain.invoke(instruction)

{'input': 'Please compute the correlation between column X1 and column X2 in table SIM_CORR_DATA_TBL, , where key is ID and maximum lag is 3.',
 'output': 'The correlation results between column X1 and column X2 with a maximum lag of 3 are stored in the table SIM_CORR_DATA_TBL_CORRELATION_RESULT.'}

In [19]:
cc.table("SIM_CORR_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF
0,-3,8.214894,0.072203,None
1,-2,2.859731,0.025135,None
2,-1,19.379410,0.170331,None
3,0,36.325105,0.319272,None
4,1,-13.231161,-0.116292,None
5,2,-14.500138,-0.127446,None
6,3,18.568941,0.163208,None


### Test Case 4 : Misconfigurated Parameter Settings

In [20]:

tool_input = dict(table_name='SIM_CORR_DATA_TBL',
                    key='ID',
                    x='X1',
                    y="X2",
                    calculate_confint=True)
cf_tool.run(tool_input=tool_input)

'ValueError occurred: Since confidence intervals are only applicable to the autocorrelation of one time-series, the value of `calculate_confint` needs to be changed to False to proceed.'

In [22]:
instruction = "Please compute the correlation between X1 and X2 in table SIM_CORR_DATA_TBL "+\
"together with the confidence intervals of the coefficients, where key is ID."
agent_chain.invoke(instruction)

{'input': 'Please compute the correlation between X1 and X2 in table SIM_CORR_DATA_TBL together with the confidence intervals of the coefficients, where key is ID.',
 'output': 'I have computed the correlation between X1 and X2 in the table SIM_CORR_DATA_TBL. The results are stored in the table SIM_CORR_DATA_TBL_CORRELATION_RESULT. Unfortunately, confidence intervals could not be calculated as they are only applicable to the autocorrelation of a single time-series.'}

In [24]:
cc.table("SIM_CORR_DATA_TBL_CORRELATION_RESULT").collect()

,LAG,CV,CF,PACF
0,-4,-6.643336,-0.058390,None
1,-3,8.214894,0.072203,None
2,-2,2.859731,0.025135,None
3,-1,19.379410,0.170331,None
4,0,36.325105,0.319272,None
5,1,-13.231161,-0.116292,None
6,2,-14.500138,-0.127446,None
7,3,18.568941,0.163208,None
8,4,-36.642442,-0.322061,None


In [25]:
instruction = "Please compute the correlation between X1 and X2 in table SIM_CORRELATION_DATA_TBL "+\
"where key is DATE."
agent_chain.invoke(instruction)

{'input': 'Please compute the correlation between X1 and X2 in table SIM_CORRELATION_DATA_TBL where key is DATE.',
 'output': 'It seems that the table "SIM_CORRELATION_DATA_TBL" could not be found. Please verify the table name and ensure that it exists in the database. If there is a typo or if the table is located in a different schema, please provide the correct details.'}

In [39]:
instruction = "Please compute the correlation between X1 and X2 in table SIM_CORR_DATA_TBL,"+\
"where key is DATE and method is dct."
agent_chain.invoke(instruction)

{'input': 'Please compute the correlation between X1 and X2 in table SIM_CORR_DATA_TBL,where key is DATE and method is dct.',
 'output': 'The method "dct" is not supported for correlation computation. The valid options for the method parameter are "auto", "brute_force", and "fft". Please choose one of these methods for the computation.'}

In [40]:
cc.drop_table("SIM_CORR_DATA_TBL_CORRELATION_RESULT")
cc.drop_table("SIM_CORR_DATA_TBL")
cc.close()